# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [ ]:
lane = "Lane 1: Ranking Signal Analysis"

why = """
I'm picking Lane 1 instead of freestyle because I want to answer the more basic
question before the harder ones: which signals in this data actually move with
visibility, clicks, or engagement at all? Lanes like Refresh Scoring or CTR
Opportunity Scoring assume I already know which signals matter and just need to
rank pages on them. I don't know that yet -- so Lane 1 (EDA, correlations, grouped
comparisons, effect sizes) is the right first step. It also plays safest with the
starter data's leakage trap: trend_direction/trend_pct are off-limits as features,
so Lane 1 lets me spend this week understanding what IS usable (position, CTR,
freshness, word count, engagement) before committing to a target and a model.
"""
print(lane)
print(why)


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [ ]:
decision = "Which content signals should a FlyRank strategist trust when deciding what to look at first?"

who_acts = """
The FlyRank content strategist / account reviewer who triages a client's page
inventory. Right now they eyeball dashboards and product flags (health_score,
priority_score) without a clear, evidence-backed sense of which raw signals those
flags are actually built on, or whether the signals behave the way the team assumes.
"""

action = """
If a signal analysis shows (for example) that CTR really does track position tier,
or that word count barely correlates with impressions, the strategist changes what
they look at first when triaging a page -- they stop treating weak signals as
priorities and start trusting the ones the data backs up.
"""

cost_of_wrong_call = """
A wrong signal claim is cheap to say and expensive to act on: if I claim a signal
matters when it doesn't (or is confounded by something else, like position), the
team wastes review hours chasing pages that were never actually a problem, and
loses trust in the whole analysis. Because Lane 1 is decision-support, not a
scored/ranked action list yet, the immediate cost is analyst hours and credibility
-- not a missed client outcome. That's exactly why every claim below has to say
"associated with" and never "causes" or "predicts."
"""

why_data_helps = """
A plain if-statement can't tell me whether an assumed relationship (e.g. "lower
position tier gets more clicks") actually holds in this data, how strong it is, or
whether it's confounded by content type or freshness. That requires actually
computing correlations, group comparisons, and effect sizes on real rows -- which
is exactly what an EDA/signal-analysis pass is for, before anyone builds a scored
model or a rule on top of an untested assumption.
"""

for name, val in [("Decision", decision), ("Who acts / what they do", action),
                   ("Cost of a wrong call", cost_of_wrong_call),
                   ("Why data/ML (not just a rule)", why_data_helps)]:
    print(f"--- {name} ---")
    print(val)


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("rows x cols:", df.shape)

# Gotcha: avg_position == 0 means "no data", not rank zero -- exclude before analysis
valid_pos = df[df["avg_position"] > 0]
no_position_data = len(df) - len(valid_pos)
print(f"\nNumber 1 -- rows with NO position data (avg_position == 0): "
      f"{no_position_data:,} ({no_position_data/len(df):.1%} of all rows). "
      f"These get dropped before any position-based signal work.")

# Number 2: does position tier actually relate to CTR? (mean CTR by position_tier)
ctr_by_tier = valid_pos.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("\nNumber 2 -- mean CTR (%) by position tier, valid rows only:")
print(ctr_by_tier.round(2))
gap = ctr_by_tier.max() / ctr_by_tier.min()
print(f"top_3 pages get ~{gap:.0f}x the mean CTR of deep pages "
      f"({ctr_by_tier.max():.2f}% vs {ctr_by_tier.min():.2f}%) -- position looks like a real signal.")

# Number 3: how much of the keyword-context data is missing, and does it follow content_type
# (the missingness gotcha from the data skill)
missing_by_type = df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean())
print("\nNumber 3 -- share of rows missing search_volume, by content_type:")
print((missing_by_type * 100).round(1).astype(str) + "%")
print("Missingness tracks content_type almost perfectly -- a blind fillna(0) would "
      "silently turn 'no keyword data' into a content-type signal.")


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
can_claim = """
WHAT THIS ANALYSIS CAN SAY:
- Observed associations: e.g. "in this 30k-row starter slice, pages in the top_3
  position tier show meaningfully higher mean CTR than pages in the deep tier."
- Directional patterns: e.g. "CTR tends to move with position tier" -- a tendency,
  not a formula.
- Decision-support signals: "this signal looks worth a strategist's attention" --
  never "this signal guarantees a result."
- Scope-limited claims: everything here is about this anonymized starter sample of
  32 clients, not FlyRank's whole client base, and not the live 79M-row warehouse.
"""

cannot_claim = """
WHAT THIS ANALYSIS CAN NEVER SAY:
- Causal claims: I cannot say a signal "causes" more clicks or visibility -- I only
  watched, I didn't run an experiment. Correlation between position and CTR is not
  proof that improving position CAUSES better CTR (confounders like content quality
  or brand exist too).
- "Predicting Google": I cannot claim to have found a Google ranking factor, or that
  I know why Google ranks anything the way it does.
- Guarantees for any single page: population-level associations don't guarantee what
  happens if one specific page is edited.
- Anything from trend_direction / trend_pct as if it were an independent finding --
  those columns define the pipeline's own decline label, so using them as "evidence"
  would just be restating the label, not discovering something new.
"""

print(can_claim)
print(cannot_claim)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.